In [3]:
# Importar dados
# ragas — avalia a qualidade do pipeline RAG
# datasets — formato que o ragas espera receber

# ── Imports ───────────────────────────────────────────────────────────
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision
)
from datasets import Dataset, load_dataset
import pandas as pd
import os
from dotenv import load_dotenv

print("✅ Imports prontos")

✅ Imports prontos


C:\Users\renat\AppData\Local\Temp\ipykernel_2724\2047165228.py:7: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\renat\AppData\Local\Temp\ipykernel_2724\2047165228.py:7: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\renat\AppData\Local\Temp\ipykernel_2724\2047165228.py:7: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (
C:\Users\renat\AppData\Local\Temp\ipykernel_2724\204716

In [4]:
# Carregar dados:
# O amnesty_qa já tem tudo que o RAGAS precisa:
# pergunta + resposta + contextos + resposta correta (ground truth)

from datasets import load_dataset

ds = load_dataset("explodinggradients/amnesty_qa", "english_v1", split="eval")

print(f"✅ {len(ds)} exemplos carregados")
print(f"Colunas disponíveis: {ds.column_names}")


✅ 20 exemplos carregados
Colunas disponíveis: ['question', 'ground_truths', 'answer', 'contexts']


In [5]:
# Ver um exemplo primeiro: perceber o que tem antes de medir
# 
# O RAGAS precisa de 4 coisas:
# 1. question — a pergunta
# 2. answer — a resposta gerada
# 3. contexts — lista de contextos recuperados
# 4. ground_truths — a resposta correcta

exemplo = ds[0]


print("PERGUNTA:")
print(exemplo["question"])
print("\nRESPOSTA:")
print(exemplo["answer"][:200], "...")
print("\nCONTEXTOS (quantos):", len(exemplo["contexts"]))
print("\nGROUND TRUTH:")
print(str(exemplo["ground_truths"])[:200])

PERGUNTA:
What are the global implications of the USA Supreme Court ruling on abortion?

RESPOSTA:
The global implications of the USA Supreme Court ruling on abortion can be significant, as it sets a precedent for other countries and influences the global discourse on reproductive rights. Here are  ...

CONTEXTOS (quantos): 3

GROUND TRUTH:
["The global implications of the USA Supreme Court ruling on abortion are significant. The ruling has led to limited or no access to abortion for one in three women and girls of reproductive age in st


In [6]:
# Preparar o dataset para o RAGAS - formato específico - converter


# Usar só os primeiros 5 exemplos para começar

# Fix: ground_truth precisa ser string
# Fix: truncar contextos para não exceder limite do Groq


n = 5  

def extrair_ground_truth(valor):
    if isinstance(valor, list):
        return " ".join(valor)
    return str(valor)

eval_dataset = Dataset.from_dict({
    # Pergunta truncada
    "question": [
        ds[i]["question"][:200]
        for i in range(n)
    ],
    # Resposta muito curta
    "answer": [
        ds[i]["answer"][:300]
        for i in range(n)
    ],
    # Só 1 contexto por exemplo, muito curto
    "contexts": [
        [ds[i]["contexts"][0][:300]]
        for i in range(n)
    ],
    # Ground truth curto
    "ground_truth": [
        extrair_ground_truth(ds[i]["ground_truths"])[:200]
        for i in range(n)
    ],
})

print(f"✅ Dataset preparado: {len(eval_dataset)} exemplos")
print(f"\nVerificação de tamanhos:")
print(f"  Pergunta:     {len(eval_dataset['question'][0])} chars")
print(f"  Resposta:     {len(eval_dataset['answer'][0])} chars")
print(f"  Contexto:     {len(eval_dataset['contexts'][0][0])} chars")
print(f"  Ground truth: {len(eval_dataset['ground_truth'][0])} chars")

✅ Dataset preparado: 5 exemplos

Verificação de tamanhos:
  Pergunta:     77 chars
  Resposta:     300 chars
  Contexto:     300 chars
  Ground truth: 200 chars


In [7]:
# Configurar LLM para o RAGAS
# O RAGAS precisa de um LLM para calcular algumas métricas (optamos pelo Groq)


from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

print("✅ LLM e embeddings prontos")
print("Modelo: llama-3.3-70b-versatile")


C:\Users\renat\AppData\Local\Temp\ipykernel_2724\475785585.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ LLM e embeddings prontos
Modelo: llama-3.3-70b-versatile


C:\Users\renat\AppData\Local\Temp\ipykernel_2724\475785585.py:20: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(llm)
C:\Users\renat\AppData\Local\Temp\ipykernel_2724\475785585.py:21: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)


In [8]:
# Configurar métricas: para cada métrica qual LLM e embeddings 

faithfulness.llm = ragas_llm
answer_relevancy.llm = ragas_llm
answer_relevancy.embeddings = ragas_embeddings
context_recall.llm = ragas_llm
context_precision.llm = ragas_llm

metricas = [faithfulness, answer_relevancy, context_recall, context_precision]

print("✅ Métricas configuradas")

✅ Métricas configuradas


In [9]:
# Rodar avaliação

# OBS: Avaliar uma métrica de cada vez para evitar sobrecarga
# e identificar qual métrica causa problema se houver erro

from ragas import RunConfig
import time

resultados_individuais = {}

for metrica in metricas:
    nome = metrica.name
    print(f"A avaliar: {nome}...")
    try:
        result = evaluate(
            dataset=eval_dataset,
            metrics=[metrica],
            run_config=RunConfig(
                max_workers=1,
                max_retries=2,
                timeout=60
            )
        )
        df_temp = result.to_pandas()
        score = df_temp[nome].mean()
        resultados_individuais[nome] = score
        print(f"  ✅ {nome}: {score:.3f}")
    except Exception as e:
        print(f"  ⚠️ {nome} falhou: {str(e)[:80]}")
        resultados_individuais[nome] = None

    # Pausa entre métricas para evitar rate limit
    time.sleep(3)

print("\n✅ Avaliação completa!")
print(resultados_individuais)

A avaliar: faithfulness...


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

  ✅ faithfulness: 0.117
A avaliar: answer_relevancy...


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

Exception raised in Job[0]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})
Exception raised in Job[1]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})
Exception raised in Job[2]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})
Exception raised in Job[3]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})
Exception raised in Job[4]: BadRequestError(Error code: 400 - {'error': {'message': "'n' : number must be at most 1", 'type': 'invalid_request_error'}})


  ✅ answer_relevancy: nan
A avaliar: context_recall...


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

  ✅ context_recall: 0.200
A avaliar: context_precision...


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

  ✅ context_precision: 0.400

✅ Avaliação completa!
{'faithfulness': np.float64(0.11666666666666665), 'answer_relevancy': nan, 'context_recall': np.float64(0.2), 'context_precision': np.float64(0.39999999996)}


In [10]:
# Resultados 

benchmarks = {
    "faithfulness": 0.80,
    "answer_relevancy": 0.80,
    "context_recall": 0.70,
    "context_precision": 0.80
}

print("SCORES MÉDIOS:")
print("=" * 45)
for nome, score in resultados_individuais.items():
    if score is not None:
        benchmark = benchmarks.get(nome, 0.80)
        status = "✅" if score >= benchmark else "⚠️"
        print(f"{nome:<25} {score:.3f}  {status}")
    else:
        print(f"{nome:<25} erro  ❌")

# Salvar 

import os
os.makedirs("../results", exist_ok=True)
pd.DataFrame([resultados_individuais]).to_csv(
    "../results/ragas_baseline.csv", index=False
)
print("\n✅ Guardado em results/ragas_baseline.csv")

SCORES MÉDIOS:
faithfulness              0.117  ⚠️
answer_relevancy          nan  ⚠️
context_recall            0.200  ⚠️
context_precision         0.400  ⚠️

✅ Guardado em results/ragas_baseline.csv
